<a href="https://colab.research.google.com/github/MJMortensonWarwick/AnalyticsEngineering2026/blob/main/2_01_schema_on_read_write.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![](https://drive.google.com/uc?export=view&id=17M9oGaOj4KdKiDDH3oTibxJ9qBTSjp6n)

# 2.01 Schema on write or schema on read?

You will load the same data into two small databases and ask the same questions of each:

| Store | Kind | Python library |
| --- | --- | --- |
| SQLite | relational (tables, SQL) | `sqlite3` (built in) |
| TinyDB | document store (JSON-like documents - NoSQL) | `tinydb` |


The point is **not speed**. It is to see two things:
1. **Schema on write versus schema on read.** *Schema on write* means the structure of the data is declared up front and the database **enforces it when data is stored**. This is the approach taken in relational databases. *Schema on read* means data is stored as it arrives and **structure is applied by whoever reads it**. This is the approach taken in most NoSQL approaches.
2. **Where the work goes.** In SQL a join is one clause and the database does it. In TinyDB there are no joins, so **you write the join yourself in Python.**

**By the end you should be able to:**
- Say which store enforced the rules, and what happened when bad data arrived.
- Translate a plain-English question into SQL and into Python code for a document store.
- Explain what the database did for you in SQLite that you had to write yourself in TinyDB.
- Give a situation where schema on read is the better choice.

**Please note**
- SQLite is **loosely typed by default**: an ordinary table will happily store the text `"12 mins"` in a number column. It enforces types only in tables created with the `STRICT` keyword, which needs a recent SQLite version (as far as I know 3.37 or later; the notebook checks your version and tells you).
- TinyDB's own documentation describes it as a "document oriented" database ([tinydb.readthedocs.io](https://tinydb.readthedocs.io/en/latest/)). It is a small library for learning and light use, so what you see here says little about large document databases. Worth noting that the difference between key value databases and document orientated databases is fairly insignificant these days.

## 1. Set up
This creates a small **synthetic bike-hire** dataset: 12 stations, 60 bikes and 2,000 trips. Eight of the bikes were never used, which matters for one of the questions.

In [ ]:
%pip install -q tinydb

In [ ]:
import sqlite3
import numpy as np
import pandas as pd
from tinydb import TinyDB, Query
from tinydb.storages import MemoryStorage

rng = np.random.default_rng(42) # set a fixed seed for random number generation. Means results are replicable.

# Dataframe of station data
stations = pd.DataFrame({
    "station_id":   [f"S{i:02d}" for i in range(1, 13)],
    "station_name": ["Market Square", "Railway Station", "Canal Basin", "University Gate", "Park Road",
                     "Library Corner", "Hospital Entrance", "Riverside Walk", "Old Town Hall",
                     "Retail Park", "Sports Centre", "Cathedral Close"],
    "zone":         ["Centre", "Centre", "North", "West", "South", "Centre",
                     "East", "South", "Centre", "North", "West", "Centre"],
    "capacity":     rng.integers(10, 31, 12),
})

# Dataframe of bike data
bikes = pd.DataFrame({
    "bike_id":       [f"B{i:03d}" for i in range(1, 61)],
    "bike_type":     rng.choice(["Standard", "Electric"], 60, p=[0.7, 0.3]),
    "purchase_year": rng.integers(2021, 2025, 60),
})

N_TRIPS = 2000 # number of trips
used_bikes = bikes["bike_id"].iloc[:52]          # the last 8 bikes are never used

# Dataframe of individual trips
trips = pd.DataFrame({
    "trip_id":          np.arange(1, N_TRIPS + 1),
    "bike_id":          rng.choice(used_bikes, N_TRIPS),
    "start_station_id": rng.choice(stations["station_id"], N_TRIPS),
    "end_station_id":   rng.choice(stations["station_id"], N_TRIPS),
    "user_type":        rng.choice(["Member", "Casual"], N_TRIPS, p=[0.65, 0.35]),
    "start_hour":       rng.integers(0, 24, N_TRIPS),
    "duration_min":     rng.gamma(2.0, 8.0, N_TRIPS).round(1),
})
print(len(stations), "stations,", len(bikes), "bikes,", len(trips), "trips")
trips.head() # print the first five rows of the trips dataframe

## 2. Same data, two stores
**Before running anything, predict.** Below are five **bad rows**. For each, will each store **accept** or **reject** it when asked to store it? Write `"accept"` or `"reject"` over the underlined string below (i.e. "____" might become "accept"). Not marked; the aim is to find where your intuition is wrong.

In [ ]:
predictions = {
    "missing duration":          {"SQLite": "___", "TinyDB": "___"},
    "duration is text":          {"SQLite": "___", "TinyDB": "___"},
    "negative duration":         {"SQLite": "___", "TinyDB": "___"},
    "station that doesn't exist": {"SQLite": "___", "TinyDB": "___"},
    "extra field not in schema": {"SQLite": "___", "TinyDB": "___"},
}

In [ ]:
GOOD = {"trip_id": 9001, "bike_id": "B001", "start_station_id": "S01", "end_station_id": "S02",
        "user_type": "Member", "start_hour": 9, "duration_min": 12.0}

# make bad variants
def variant(**changes):
    doc = dict(GOOD); doc.update(changes); return doc

bad_docs = {
    "missing duration":          {k: v for k, v in GOOD.items() if k != "duration_min"} | {"trip_id": 9001},
    "duration is text":          variant(trip_id=9002, duration_min="12 mins"),
    "negative duration":         variant(trip_id=9003, duration_min=-5.0),
    "station that doesn't exist": variant(trip_id=9004, start_station_id="S99"),
    "extra field not in schema": variant(trip_id=9005, notes="bike returned damaged"),
}

### 2a. SQLite: schema on write
Read the `CREATE TABLE` statements. The rules live **in the database**: types, `NOT NULL`, allowed values (`CHECK`), and `REFERENCES` (foreign keys, meaning "this must match a row in another table").

In [ ]:
print("SQLite version:", sqlite3.sqlite_version)
STRICT = " STRICT" if sqlite3.sqlite_version_info >= (3, 37, 0) else ""
if not STRICT:
    print("This SQLite is older than 3.37, so column types are NOT enforced. Expect the 'duration is text' row to be accepted.")

sql_con = sqlite3.connect(":memory:")
sql_con.execute("PRAGMA foreign_keys = ON")        # SQLite only checks foreign keys if you switch this on
sql_con.executescript(f"""
CREATE TABLE stations (
    station_id   TEXT PRIMARY KEY,
    station_name TEXT NOT NULL,
    zone         TEXT NOT NULL,
    capacity     INTEGER NOT NULL
){STRICT};

CREATE TABLE bikes (
    bike_id       TEXT PRIMARY KEY,
    bike_type     TEXT NOT NULL CHECK (bike_type IN ('Standard', 'Electric')),
    purchase_year INTEGER NOT NULL
){STRICT};

CREATE TABLE trips (
    trip_id          INTEGER PRIMARY KEY,
    bike_id          TEXT NOT NULL REFERENCES bikes(bike_id),
    start_station_id TEXT NOT NULL REFERENCES stations(station_id),
    end_station_id   TEXT NOT NULL REFERENCES stations(station_id),
    user_type        TEXT NOT NULL CHECK (user_type IN ('Member', 'Casual')),
    start_hour       INTEGER NOT NULL CHECK (start_hour BETWEEN 0 AND 23),
    duration_min     REAL NOT NULL CHECK (duration_min > 0)
){STRICT};
""")

for name, frame in [("stations", stations), ("bikes", bikes), ("trips", trips)]:
    marks = ",".join("?" * frame.shape[1])
    sql_con.executemany(f"INSERT INTO {name} VALUES ({marks})", list(frame.itertuples(index=False, name=None)))
sql_con.commit()
print({t: sql_con.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0] for t in ["stations", "bikes", "trips"]})

Now try to store the five bad rows in SQLite. Any row that gets in is removed again so the data stays clean.

In [ ]:
def try_insert_sqlite(doc):
    cols = list(doc)
    sql = f"INSERT INTO trips ({', '.join(cols)}) VALUES ({', '.join('?' * len(cols))})"
    try:
        sql_con.execute(sql, [doc[c] for c in cols])
        sql_con.execute("DELETE FROM trips WHERE trip_id = ?", (doc["trip_id"],))   # keep the data clean
        sql_con.commit()
        return "accept", ""
    except sqlite3.Error as e:
        return "reject", str(e)

sqlite_outcomes = {}
for name, doc in bad_docs.items():
    result, msg = try_insert_sqlite(doc)
    sqlite_outcomes[name] = result
    print(f"{name:28s} SQLite {result.upper():7s} {msg}")

### 2b. TinyDB: schema on read
TinyDB has no `CREATE TABLE`. We just hand it dictionaries. It does have named tables (`db.table("trips")`) so we keep the same three groups of data. We use in-memory storage so there is no file.

We use a **second copy** (`scratch`) for the bad rows so the main data stays clean.

In [ ]:
def build_tiny():
    db = TinyDB(storage=MemoryStorage)
    db.table("stations").insert_multiple(stations.to_dict("records"))
    db.table("bikes").insert_multiple(bikes.to_dict("records"))
    db.table("trips").insert_multiple(trips.to_dict("records"))
    return db

tiny_db = build_tiny()
print({t: len(tiny_db.table(t)) for t in ["stations", "bikes", "trips"]})
print(tiny_db.table("stations").all()[0])

In [ ]:
scratch = build_tiny()
tiny_outcomes = {}
for name, doc in bad_docs.items():
    scratch.table("trips").insert(doc)
    tiny_outcomes[name] = "accept"
    print(f"{name:28s} TinyDB accepted it")
print("trips in scratch:", len(scratch.table("trips")), "(was 2000)")

In [ ]:
check = pd.DataFrame({
    "you predicted: SQLite": {k: v["SQLite"] for k, v in predictions.items()},
    "SQLite did":            sqlite_outcomes,
    "you predicted: TinyDB": {k: v["TinyDB"] for k, v in predictions.items()},
    "TinyDB did":            tiny_outcomes,
})
check

**The accepted rows are not harmless.** TinyDB accepted them, but any code that reads them must cope. Here is a simple "average duration by user type" written for TinyDB. Run it on the clean data and then on the scratch copy:

In [ ]:
def naive_average_by_user_type(db):
    total, count = {}, {}
    for t in db.table("trips"):
        u = t["user_type"]
        total[u] = total.get(u, 0) + t["duration_min"]
        count[u] = count.get(u, 0) + 1
    return {u: round(total[u] / count[u], 3) for u in total}

print("on the clean data:", naive_average_by_user_type(tiny_db))
try:
    print(naive_average_by_user_type(scratch))
except Exception as e:
    print("on the scratch data it failed:", type(e).__name__, e)

Nothing was wrong with the *code* when the data was clean. The schema was never written down anywhere, so the first bad row broke it, and each reader of this data has to defend against such rows separately.

**Short answer cell:** In one or two sentences, where does the schema live in SQLite, and where does it live in TinyDB?

*Your answer:*

## 3. The same question, in English, SQL and Python
Now the main exercise. Each question is stated **in plain English first**. Your job is to translate it twice: into **SQL** for SQLite, and into **Python** for TinyDB. Both stores hold the same clean data, so the answers must match. A check cell after each question tells you.

Tips for TinyDB:
- `db.table("trips")` gives a table you can loop over (`for t in table:`) or search (`table.search(Query().user_type == "Casual")`, or `table.count(...)`).
- There is **no join and no group-by**. Use dictionaries for lookups and running totals.
- Return results in the exact shape stated so the check can compare them.

Helper used by the checks:

In [ ]:
def compare(label, from_sqlite, from_tiny):
    same = from_sqlite == from_tiny
    print(f"{label}: {'the two stores AGREE' if same else 'the two stores DIFFER'}")
    print("  SQLite:", from_sqlite)
    print("  TinyDB:", from_tiny)

### Q1. *"How many trips were made by Casual users?"* (worked example, run as is)
One table, so it is almost the same in both. Returns a number.

In [ ]:
def q1_sqlite():
    return sql_con.execute("SELECT COUNT(*) FROM trips WHERE user_type = 'Casual'").fetchone()[0]

def q1_tiny(db=tiny_db):
    return db.table("trips").count(Query().user_type == "Casual")

compare("Q1", q1_sqlite(), q1_tiny())

### Q2. *"What is the average trip duration, in minutes, for trips starting in each zone?"*
Needs **two tables**: the trips (which have a start station id) and the stations (which know the zone). Returns a dictionary `{zone: average rounded to 3 decimals}`.

The SQLite version is given. **You write the TinyDB version.** Hints: build a dictionary `{station_id: zone}` from the stations table, loop over the trips, and keep a running total and count for each zone.

In [ ]:
def q2_sqlite():
    rows = sql_con.execute("""
        SELECT s.zone, AVG(t.duration_min)
        FROM trips t
        JOIN stations s ON t.start_station_id = s.station_id
        GROUP BY s.zone""").fetchall()
    return {zone: round(avg, 3) for zone, avg in rows}

In [ ]:
def q2_tiny(db=tiny_db):
    # TODO: look up each trip's zone, then average the duration per zone
    ...

compare("Q2", q2_sqlite(), q2_tiny())

### Q3. *"How many trips by Electric bikes ended in each zone?"*
Needs **three tables**: trips, bikes (for the bike type) and stations (for the zone of the **end** station). Returns `{zone: number_of_trips}`.

You write **both** versions.

In [ ]:
def q3_sqlite():
    # TODO: join trips to bikes and to stations (using end_station_id), filter, group, count
    ...

def q3_tiny(db=tiny_db):
    # TODO: build two lookup dictionaries (bike -> type, station -> zone), then loop over trips
    ...

compare("Q3", q3_sqlite(), q3_tiny())

### Q4. *"Which bikes (id and type) have never been used for a trip?"*
Something that is **not in** another table. Returns a list of `(bike_id, bike_type)` tuples sorted by bike id.

SQL hint: a `LEFT JOIN` from bikes to trips keeps bikes with no matching trip; those rows have `NULL` in the trip columns. TinyDB hint: collect the set of bike ids that appear in trips, then keep the bikes not in that set.

In [ ]:
def q4_sqlite():
    # TODO
    ...

def q4_tiny(db=tiny_db):
    # TODO
    ...

compare("Q4", q4_sqlite(), q4_tiny())

## 4. Reading defensively: schema on read in practice
In Part 2 the naive TinyDB code crashed on the scratch data. SQLite never had to deal with those rows, because it refused them on the way in.

Write a safe version of Q2 for TinyDB that applies, at read time, the rules SQLite applied at write time. It should skip any trip where:
- `duration_min` is missing, or
- `duration_min` is not a number (`int` or `float`), or
- `duration_min` is zero or negative, or
- the start station does not exist.

Return two things: the same dictionary as Q2, and a list of `(trip_id, reason)` for each skipped trip.

In [ ]:
def q2_tiny_safe(db):
    skipped = []
    # TODO: like q2_tiny, but check each trip against the four rules above and record why it was skipped
    ...
    return result, skipped

result, skipped = q2_tiny_safe(scratch)
print("result :", result)
print("skipped:", skipped)

Four rows were skipped, and the fifth bad row (the one with the extra `notes` field) was **kept**, because its duration and station were fine. SQLite refused that row; TinyDB accepted it and your code counted it, so a zone's average may differ slightly.

Was that the right outcome? There is no technical answer. It is an

![](https://drive.google.com/uc?export=view&id=1JjZejnp5SGPBXChOr2vHWmVQJDWq7HeQ)

(i.e. a business decision) about what counts as a valid trip.

In SQLite someone made that decision once and wrote it into the schema. In TinyDB every reader has to make it again.

## 5. Your own question
Write a question of your own, **in plain English**, that needs **at least one join**. Then translate it into SQL and into Python and check the two agree. Some ideas:
- Which station is the most common start point for Casual users in the Centre zone?
- What is the average trip duration for Electric versus Standard bikes?
- Which zone has the most trips that start and end in the same zone?

In [ ]:
my_question = "Write your question here, in plain English"

def mine_sqlite():
    # TODO
    ...

def mine_tiny(db=tiny_db):
    # TODO
    ...

compare(my_question, mine_sqlite(), mine_tiny())

## 6. Reflection
1. In SQLite, what did the *database* do for you in Part 2? What did you have to write *yourself* in Part 4 to get similar protection in TinyDB?
2. Which of Q1 to Q4 was the most different to write in the two systems, and why? What did you have to decide in TinyDB that SQL decided for you?
3. Suppose a second analyst writes their own TinyDB code for Q2 and forgets to skip the bad rows. What happens to the team's numbers?
4. Give two situations where schema on read is the better choice. How does this connect to a data lake or the bronze layer (week 2)?
5. What is the cost of schema on write? What would you need to do in SQLite to accept the extra `notes` field legitimately?